In [14]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# File paths
ALL_CHUNKS_PATH = Path("all_chunks.csv")
ANNOTATIONS_PATH = Path("Monday_20260223_Kvant1_lab_transcript_broad_annotations.csv")
# ALL_CHUNKS_PATH = Path("../all_chunks.csv")

In [15]:
all_chunks = pd.read_csv(ALL_CHUNKS_PATH)
annotations = pd.read_csv(ANNOTATIONS_PATH)

print("all_chunks shape:", all_chunks.shape)
print("annotations shape:", annotations.shape)

print("\nAnnotation columns:")
print(annotations.columns.tolist())

all_chunks shape: (66, 8)
annotations shape: (9, 15)

Annotation columns:
['interview_id', 'chunk_id', 'speaker', 'speaker_type', 'participant_id', 'text', 'previous_context', 'next_context', 'broad_topics', 'confidence', 'theory_cues', 'annotation_reason', 'valence', 'possible_tension', 'human_review_needed']


In [16]:
def to_bool(x):
    if pd.isna(x):
        return False
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in ["true", "1", "yes", "y"]

annotations["human_review_needed_clean"] = annotations["human_review_needed"].apply(to_bool)

print("Number of chunks:", len(annotations))
print("Chunks needing human review:", annotations["human_review_needed_clean"])
print("Chunks not needing human review:", (~annotations["human_review_needed_clean"]).sum())

Number of chunks: 9
Chunks needing human review: 0     True
1    False
2     True
3     True
4     True
5     True
6     True
7    False
8    False
Name: human_review_needed_clean, dtype: bool
Chunks not needing human review: 3


In [17]:
review_df = annotations[annotations["human_review_needed_clean"]].copy()

cols_to_show = [
    "interview_id",
    "chunk_id",
    "speaker",
    "speaker_type",
    "participant_id",
    "broad_topics",
    "confidence",
    "theory_cues",
    "annotation_reason",
    "valence",
    "possible_tension",
    "human_review_needed",
    "text"
]

cols_to_show = [c for c in cols_to_show if c in review_df.columns]

In [18]:
human_review_summary = review_df[[
    "chunk_id",
    "confidence",
    "broad_topics",
    "theory_cues",
    "annotation_reason",
    "possible_tension",
    "text"
]].copy()

# Shorten text for easier table viewing
human_review_summary["short_text"] = human_review_summary["text"].astype(str).str.slice(0, 500)

display(
    human_review_summary[
        [
            "chunk_id",
            "confidence",
            "broad_topics",
            "theory_cues",
            "annotation_reason",
            "possible_tension",
            "short_text"
        ]
    ]
)
for _, row in review_df.iterrows():
    display(Markdown(f"## Human review needed: `{row['chunk_id']}`"))

    display(Markdown(f"**Confidence:** {row.get('confidence', 'NA')}"))
    display(Markdown(f"**Broad topics:** {row.get('broad_topics', 'NA')}"))
    display(Markdown(f"**Theory cues:** {row.get('theory_cues', 'NA')}"))
    display(Markdown(f"**Valence:** {row.get('valence', 'NA')}"))

    display(Markdown("### Annotation reason"))
    display(Markdown(str(row.get("annotation_reason", ""))))

    display(Markdown("### Possible tension / reason for review"))
    display(Markdown(str(row.get("possible_tension", ""))))

    display(Markdown("### Chunk text"))
    display(Markdown(f"> {str(row.get('text', '')).replace(chr(10), '<br>')}"))

    display(Markdown("---"))



for _, row in review_df.iterrows():
    display(Markdown(f"## Context review: `{row['chunk_id']}`"))

    previous_context = row.get("previous_context", "")
    text = row.get("text", "")
    next_context = row.get("next_context", "")

    display(Markdown("### Previous context"))
    display(Markdown(f"> {str(previous_context).replace(chr(10), '<br>')}"))

    display(Markdown("### Main chunk"))
    display(Markdown(f"> **{str(text).replace(chr(10), '<br>')}**"))

    display(Markdown("### Next context"))
    display(Markdown(f"> {str(next_context).replace(chr(10), '<br>')}"))

    display(Markdown("### Why AI flagged this"))
    display(Markdown(f"**Possible tension:** {row.get('possible_tension', '')}"))

    display(Markdown("---"))

annotations["confidence_numeric"] = pd.to_numeric(annotations["confidence"], errors="coerce")

low_confidence = annotations[
    annotations["confidence_numeric"].notna() &
    (annotations["confidence_numeric"] < 0.80)
].copy()

print("Low-confidence chunks:", len(low_confidence))

display(
    low_confidence[
        [
            "chunk_id",
            "confidence",
            "broad_topics",
            "theory_cues",
            "annotation_reason",
            "possible_tension",
            "human_review_needed",
            "text"
        ]
    ]
)


,chunk_id,confidence,broad_topics,theory_cues,annotation_reason,possible_tension,short_text
0,Monday_20260223_Kvant1_lab_transcript_C0001,0.75,SELF_DETERMINATION; OTHER_RELEVANT; CONTEXT_ONLY,Autonomy supported; Ownership; Controlled moti...,Student 1 contrasts the current free-play lab ...,The word 'messing around' is ambiguous — it co...,"Monday 20260223 Kvant1 lab.m4a, student 1: gir..."
2,Monday_20260223_Kvant1_lab_transcript_C0003,0.80,SELF_DETERMINATION; VALUE_MOTIVATION; SELF_EFF...,Emotional cost; Time cost; Competence frustrat...,Students describe how stress from data collect...,The chunk mixes descriptions of other labs (ne...,Having to take so much data in some of the lab...
3,Monday_20260223_Kvant1_lab_transcript_C0004,0.80,SELF_EFFICACY; SELF_DETERMINATION; VALUE_MOTIV...,Theory-lab connection self-efficacy; Competenc...,Student 2 expresses difficulty understanding p...,Students with optics background feel competent...,"[laughs]\n\nstudent 2\nI think for my part, ac..."
4,Monday_20260223_Kvant1_lab_transcript_C0005,0.50,SELF_EFFICACY; SELF_DETERMINATION,Peer modelling; Relatedness supported; Compete...,Student 3 mentions not fully understanding the...,The student did not fully grasp the content bu...,So I didn't really get it totally but then it ...
5,Monday_20260223_Kvant1_lab_transcript_C0006,0.72,SELF_EFFICACY; VALUE_MOTIVATION; OTHER_RELEVANT,Equipment/experimental self-efficacy; Expectan...,Student 2 describes expecting the optical setu...,Student explicitly labels the informal setup a...,co students who has a are following the optics...
6,Monday_20260223_Kvant1_lab_transcript_C0007,0.78,VALUE_MOTIVATION; SELF_DETERMINATION,Intrinsic value; Low perceived value; Value sh...,Students express that the experiment was less ...,Students hedge on whether they are truly disap...,In some way. Bjarke\nSuper. Anything else? stu...


## Human review needed: `Monday_20260223_Kvant1_lab_transcript_C0001`

**Confidence:** 0.75

**Broad topics:** SELF_DETERMINATION; OTHER_RELEVANT; CONTEXT_ONLY

**Theory cues:** Autonomy supported; Ownership; Controlled motivation contrast; Lab format details

...

### Why AI flagged this

**Possible tension:** Students hedge on whether they are truly disappointed, suggesting the negative valence is mild or ambiguous; the interviewer's framing of 'disappointment' may have influenced responses.

---

Low-confidence chunks: 4


,chunk_id,confidence,broad_topics,theory_cues,annotation_reason,possible_tension,human_review_needed,text
0,Monday_20260223_Kvant1_lab_transcript_C0001,0.75,SELF_DETERMINATION; OTHER_RELEVANT; CONTEXT_ONLY,Autonomy supported; Ownership; Controlled moti...,Student 1 contrasts the current free-play lab ...,The word 'messing around' is ambiguous — it co...,True,"Monday 20260223 Kvant1 lab.m4a, student 1: gir..."
4,Monday_20260223_Kvant1_lab_transcript_C0005,0.50,SELF_EFFICACY; SELF_DETERMINATION,Peer modelling; Relatedness supported; Compete...,Student 3 mentions not fully understanding the...,The student did not fully grasp the content bu...,True,So I didn't really get it totally but then it ...
5,Monday_20260223_Kvant1_lab_transcript_C0006,0.72,SELF_EFFICACY; VALUE_MOTIVATION; OTHER_RELEVANT,Equipment/experimental self-efficacy; Expectan...,Student 2 describes expecting the optical setu...,Student explicitly labels the informal setup a...,True,co students who has a are following the optics...
6,Monday_20260223_Kvant1_lab_transcript_C0007,0.78,VALUE_MOTIVATION; SELF_DETERMINATION,Intrinsic value; Low perceived value; Value sh...,Students express that the experiment was less ...,Students hedge on whether they are truly disap...,True,In some way. Bjarke\nSuper. Anything else? stu...
